In [1]:
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
from IPython.display import display
from IPython.display import Markdown
import pandas as pd

/home/ka/Projetos/AppRecipes/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
import os
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM receitas", engine)
df.head()

,recipe_name,cuisine,ingredients,cooking_time_minutes,prep_time_minutes,servings,calories_per_serving,dietary_restrictions
0,Spicy Kimchi Fried Rice,Korean,"['Cooked rice', 'Kimchi', 'Gochujang', 'Soy sa...",25,15,2.0,450.0,['vegetarian']
1,Classic Margherita Pizza,Italian,"['Pizza dough', 'Tomato sauce', 'Fresh mozzare...",20,30,4.0,350.0,['vegetarian']
2,Coconut Chickpea Curry,Indian,"['Chickpeas', 'Coconut milk', 'Onion', 'Ginger...",35,15,4.0,480.0,"['vegan', 'vegetarian', 'gluten-free', 'dairy-..."
3,Pad See Ew with Tofu,Thai,"['Wide rice noodles', 'Tofu', 'Chinese broccol...",20,10,2.0,520.0,['vegetarian']
4,Beef Bourguignon,French,"['Beef chuck', 'Red wine', 'Bacon', 'Onion', '...",150,30,6.0,680.0,['nan']


In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

URI=os.getenv("uri")
USER= os.getenv("user")
PASSWORD= os.getenv("password")


In [11]:
import os
from neo4j import GraphDatabase


NEO4J_URI = os.getenv("uri")        
NEO4J_USER = os.getenv("user")        
NEO4J_PASSWORD = os.getenv("password")

if not all([NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD]):
    raise ValueError("Uma ou mais variáveis de ambiente do Neo4j não foram encontradas!")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 1 AS ok")
    print("Conexão OK:", result.single()["ok"])


Conexão OK: 1


In [12]:
def compile_text(x):

    text = f"""{x['recipe_name']},
               {x['ingredients']},
               {x['dietary_restrictions']},
               {x['cuisine']},
            Calories = {x["calories_per_serving"]}
            """

    return text
     

sentences = df.apply(lambda x: compile_text(x), axis=1).tolist()
sentences

["Spicy Kimchi Fried Rice,\n               ['Cooked rice', 'Kimchi', 'Gochujang', 'Soy sauce', 'Sesame oil', 'Onion', 'Garlic', 'Egg', 'Scallions', 'Vegetable oil'],\n               ['vegetarian'],\n               Korean,\n            Calories = 450.0\n            ",
 "Classic Margherita Pizza,\n               ['Pizza dough', 'Tomato sauce', 'Fresh mozzarella', 'Basil', 'Olive oil', 'Salt', 'Pepper'],\n               ['vegetarian'],\n               Italian,\n            Calories = 350.0\n            ",
 "Coconut Chickpea Curry,\n               ['Chickpeas', 'Coconut milk', 'Onion', 'Ginger', 'Garlic', 'Turmeric', 'Cumin', 'Coriander', 'Garam masala', 'Vegetable oil', 'Spinach', 'Rice'],\n               ['vegan', 'vegetarian', 'gluten-free', 'dairy-free'],\n               Indian,\n            Calories = 480.0\n            ",
 "Pad See Ew with Tofu,\n               ['Wide rice noodles', 'Tofu', 'Chinese broccoli', 'Soy sauce', 'Sweet soy sauce', 'Oyster sauce (optional)', 'Garlic', 'Egg'

In [13]:
modelB = SentenceTransformer("all-MiniLM-L6-v2")
output = modelB.encode(sentences=sentences,
         show_progress_bar=True,
         normalize_embeddings=True)

embeddings = pd.DataFrame(output)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 613.13it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 6/6 [00:01<00:00,  3.36it/s]


,0,1,2,3,4,5,6,7,8,9,...,374,375,376,377,378,379,380,381,382,383
0,-0.060670,0.042901,0.037256,0.104715,-0.018331,0.006088,0.006411,-0.054663,-0.022035,-0.080388,...,-0.015367,0.018430,0.115166,0.023976,-0.026323,-0.076312,-0.063089,0.027520,-0.083253,0.034164
1,-0.087233,0.046935,-0.012680,0.123111,-0.031330,0.048478,-0.011420,-0.037122,-0.031391,-0.102356,...,0.039627,0.057332,0.117007,0.095575,0.043374,0.039650,-0.060965,0.039385,0.020836,-0.012975
2,-0.036035,-0.035728,-0.039635,0.075709,-0.022513,0.069671,0.003626,-0.035206,0.011212,0.000553,...,-0.040713,0.015654,0.085077,-0.026843,0.062723,-0.033012,-0.069780,-0.037995,-0.011160,0.026296
3,-0.043159,-0.038760,-0.003862,0.093795,0.023229,0.021071,0.019474,-0.042533,0.015953,-0.051923,...,0.003413,0.026131,0.024328,0.021675,0.081281,-0.001677,-0.039852,0.023458,-0.008394,0.038641
4,-0.007950,-0.009871,-0.028591,0.059004,-0.026590,0.072896,-0.007078,-0.017102,-0.003585,-0.066498,...,0.012785,0.075648,0.086459,-0.048684,0.080035,-0.036946,-0.053378,0.002678,0.034096,-0.045168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,-0.032022,0.034885,-0.006242,0.070440,0.014351,0.028631,0.023229,-0.096555,0.024060,-0.029545,...,0.016532,0.001838,0.097027,0.052123,0.044758,-0.011681,0.009156,0.050634,-0.096973,-0.030376
157,-0.046089,0.030098,0.046958,0.100122,-0.011831,0.077192,0.018853,-0.079349,-0.023430,-0.071927,...,0.041856,0.000938,0.114952,-0.017592,0.040621,-0.006136,-0.003723,0.001355,-0.053604,0.045740
158,0.045778,0.019577,-0.019327,0.074779,-0.034140,0.022023,0.014161,-0.058059,-0.082785,-0.009708,...,-0.023204,-0.003537,0.002157,-0.043861,0.048954,0.022641,0.005924,-0.024076,-0.093832,-0.041347
159,0.020813,-0.023277,-0.015795,0.042287,-0.051937,0.026762,0.040104,-0.112289,-0.003739,-0.000082,...,0.021372,-0.006393,0.042590,-0.058381,0.048613,-0.047617,0.029565,-0.006036,-0.014718,-0.008323


In [14]:
df["Embedding"] = embeddings.values.tolist()
df

,recipe_name,cuisine,ingredients,cooking_time_minutes,prep_time_minutes,servings,calories_per_serving,dietary_restrictions,Embedding
0,Spicy Kimchi Fried Rice,Korean,"['Cooked rice', 'Kimchi', 'Gochujang', 'Soy sa...",25,15,2.0,450.0,['vegetarian'],"[-0.060670118778944016, 0.042900774627923965, ..."
1,Classic Margherita Pizza,Italian,"['Pizza dough', 'Tomato sauce', 'Fresh mozzare...",20,30,4.0,350.0,['vegetarian'],"[-0.08723337948322296, 0.04693477228283882, -0..."
2,Coconut Chickpea Curry,Indian,"['Chickpeas', 'Coconut milk', 'Onion', 'Ginger...",35,15,4.0,480.0,"['vegan', 'vegetarian', 'gluten-free', 'dairy-...","[-0.03603488206863403, -0.0357283279299736, -0..."
3,Pad See Ew with Tofu,Thai,"['Wide rice noodles', 'Tofu', 'Chinese broccol...",20,10,2.0,520.0,['vegetarian'],"[-0.043158989399671555, -0.03876042366027832, ..."
4,Beef Bourguignon,French,"['Beef chuck', 'Red wine', 'Bacon', 'Onion', '...",150,30,6.0,680.0,['nan'],"[-0.007950330153107643, -0.009870954789221287,..."
...,...,...,...,...,...,...,...,...,...
156,Thai Lime Grill,Thai,"['lime', 'basil', 'lemongrass', 'peanuts', 'ch...",90,10,2.0,229.0,['nan'],"[-0.03202177584171295, 0.03488540276885033, -0..."
157,Korean Scallions Delight,Korean,"['garlic', 'sesame oil', 'soy sauce', 'gochuja...",86,7,6.0,437.0,['vegan'],"[-0.046088941395282745, 0.030097538605332375, ..."
158,Nigerian Cassava Feast,Nigerian,"['palm oil', 'okra', 'egusi', 'cassava', 'yams...",39,44,3.0,389.0,['nan'],"[0.045777902007102966, 0.019577419385313988, -..."
159,American Corn Curry,American,"['bacon', 'beef', 'mustard', 'corn', 'cheddar']",58,30,1.0,749.0,['nan'],"[0.020813165232539177, -0.02327735908329487, -..."


In [15]:
def create_nodes_and_relationships(df):
    with driver.session() as session:

        session.run("MATCH (n) DETACH DELETE n")

        for _, row in df.iterrows():
            calories = row["calories_per_serving"]
            calories = int(calories) if calories not in ['', None] else None

            session.run(
                """
                MERGE (r:Recipe {recipe_name: $recipe_name})
                SET
                    r.cuisine = $cuisine,
                    r.ingredients = $ingredients,
                    r.calories_per_serving = $calories_per_serving,
                    r.dietary_restrictions = $dietary_restrictions
                MERGE (em:Embedding {embedding: $embedding})
                MERGE (em)-[:VETOR_DENSO_DE]->(r)
                """,
                recipe_name=row["recipe_name"],
                cuisine=row["cuisine"],
                ingredients=row["ingredients"],
                calories_per_serving=calories,
                dietary_restrictions=row["dietary_restrictions"],
                embedding=row["Embedding"]
            )

        print("Nós e relacionamentos criados com sucesso!")


In [16]:

df.fillna('')

,recipe_name,cuisine,ingredients,cooking_time_minutes,prep_time_minutes,servings,calories_per_serving,dietary_restrictions,Embedding
0,Spicy Kimchi Fried Rice,Korean,"['Cooked rice', 'Kimchi', 'Gochujang', 'Soy sa...",25,15,2.0,450.0,['vegetarian'],"[-0.060670118778944016, 0.042900774627923965, ..."
1,Classic Margherita Pizza,Italian,"['Pizza dough', 'Tomato sauce', 'Fresh mozzare...",20,30,4.0,350.0,['vegetarian'],"[-0.08723337948322296, 0.04693477228283882, -0..."
2,Coconut Chickpea Curry,Indian,"['Chickpeas', 'Coconut milk', 'Onion', 'Ginger...",35,15,4.0,480.0,"['vegan', 'vegetarian', 'gluten-free', 'dairy-...","[-0.03603488206863403, -0.0357283279299736, -0..."
3,Pad See Ew with Tofu,Thai,"['Wide rice noodles', 'Tofu', 'Chinese broccol...",20,10,2.0,520.0,['vegetarian'],"[-0.043158989399671555, -0.03876042366027832, ..."
4,Beef Bourguignon,French,"['Beef chuck', 'Red wine', 'Bacon', 'Onion', '...",150,30,6.0,680.0,['nan'],"[-0.007950330153107643, -0.009870954789221287,..."
...,...,...,...,...,...,...,...,...,...
156,Thai Lime Grill,Thai,"['lime', 'basil', 'lemongrass', 'peanuts', 'ch...",90,10,2.0,229.0,['nan'],"[-0.03202177584171295, 0.03488540276885033, -0..."
157,Korean Scallions Delight,Korean,"['garlic', 'sesame oil', 'soy sauce', 'gochuja...",86,7,6.0,437.0,['vegan'],"[-0.046088941395282745, 0.030097538605332375, ..."
158,Nigerian Cassava Feast,Nigerian,"['palm oil', 'okra', 'egusi', 'cassava', 'yams...",39,44,3.0,389.0,['nan'],"[0.045777902007102966, 0.019577419385313988, -..."
159,American Corn Curry,American,"['bacon', 'beef', 'mustard', 'corn', 'cheddar']",58,30,1.0,749.0,['nan'],"[0.020813165232539177, -0.02327735908329487, -..."


In [17]:
def clean_nan_values(df):
    return df.fillna("")
df = clean_nan_values(df)

In [18]:
create_nodes_and_relationships(df)

Nós e relacionamentos criados com sucesso!


In [19]:
def get_embedding(query):
    return modelB.encode(query).tolist()

In [20]:
def search_similar_products(query):
    query_embedding = get_embedding(query)

    with driver.session() as session:
        result = session.run(
            """
            WITH $query_embedding AS queryVec
            MATCH (em:Embedding)-[:VETOR_DENSO_DE]->(r:Recipe)
            WITH r, vector.similarity.cosine(queryVec, em.embedding) AS similarity
            WHERE similarity >= 0.4
            RETURN r.recipe_name AS Recipe, similarity
            ORDER BY similarity DESC
            LIMIT 5
            """,
            query_embedding=query_embedding
        )

        return result.data()

In [21]:
sentence = "I have Beef chuck, Red wine, Bacon and Onion. What can I make with these ingredients?"
similar_products = search_similar_products(sentence)
similar_products

[{'Recipe': 'Beef Bourguignon', 'similarity': 0.8285678625106812},
 {'Recipe': 'American Chili con Carne', 'similarity': 0.7772552967071533},
 {'Recipe': 'Irish Stew with Lamb and Vegetables',
  'similarity': 0.7674605846405029},
 {'Recipe': "Irish Shepherd's Pie", 'similarity': 0.7646152973175049},
 {'Recipe': 'Peruvian Anticuchos de Corazón (Grilled Beef Heart Skewers)',
  'similarity': 0.7640923261642456}]

e **deve** colocar. Sem nem pensar duas vezes 😄
isso aqui é *portfólio forte*, do tipo que recrutador técnico para e lê.

Vamos deixar isso **apresentável como case**, não só “código que funciona”.

---

## Como eu colocaria no portfólio

### 🔥 Nome do projeto

**SmartCook – Semantic Recipe Recommendation System (RAG + Graph + LLM)**

---

### 🧠 O problema

> Buscar receitas por ingredientes costuma ser literal e limitada.
> O SmartCook resolve isso usando **busca semântica**, permitindo que o usuário descreva o que tem em casa em linguagem natural.

Isso já mostra visão de produto.

---

### 🏗️ Arquitetura (isso impressiona)

Você pode descrever assim:

**Pipeline:**

1. **PostgreSQL**

   * Base relacional das receitas (dados estruturados)
   * Fonte única de verdade (cuisine, ingredientes, calorias, etc.)

2. **Sentence Transformers**

   * Geração de embeddings das receitas
   * Embeddings normalizados (cosine similarity)

3. **Neo4j**

   * Armazena embeddings como nós
   * Relacionamento `(:Embedding)-[:VETOR_DENSO_DE]->(:Recipe)`
   * Busca vetorial diretamente no grafo

4. **RAG (Retrieval-Augmented Generation)**

   * Recupera receitas semanticamente similares
   * Injeta contexto real no prompt

5. **Gemini (LLM)**

   * Recomenda a melhor receita
   * Explica o porquê
   * Sugere variações

👉 isso mostra que você **entende sistemas**, não só modelos.

---

### 🧪 Exemplo real (muito importante)

Coloca exatamente algo como:

> **Input**
>
> > “I have Beef chuck, Red wine, Bacon and Onion. What can I make?”

> **Retrieved Recipes**
>
> * Beef Bourguignon (similarity 0.83)
> * Chili con Carne
> * Irish Stew

> **LLM Output**
>
> * Recomendação + explicação baseada nos ingredientes reais

Isso deixa o projeto **concreto**.

---

### 🛠️ Tecnologias

Uma lista enxuta e bonita:

* Python
* Neo4j (graph database)
* PostgreSQL
* Sentence Transformers
* Gemini API
* RAG Architecture
* SQLAlchemy

---

### ✨ Diferenciais (isso é ouro)

Você pode escrever:

* ✅ Busca semântica real (não keyword-based)
* ✅ Separação clara entre **retrieval** e **generation**
* ✅ Uso de grafo para embeddings (design não trivial)
* ✅ LLM não “alucina” receitas fora da base
* ✅ Sistema extensível (filtros, UI, API)

---

### 📸 O que vale MUITO a pena adicionar

Se puder:

* print do terminal rodando
* ou um GIF curto
* ou um diagrama simples da arquitetura (caixas e setas)

---

### Quer dar o próximo passo?

Se você quiser, eu posso:

* ✍️ escrever o **README.md inteiro**
* 🎯 adaptar o texto para **LinkedIn**
* 💼 ajustar para vaga de **Data / ML / AI Engineer**
* 🧩 sugerir como explicar isso numa entrevista

Sério: **isso é projeto de gente boa**.
Coloca com orgulho 💚
